In [1]:
#!git clone https://github.com/nrimsky/CAA.git

In [2]:
#%cd CAA

In [3]:
import pandas as pd
import json
import random
import numpy as np

# 读取数据集
dataset = pd.read_parquet("hf://datasets/alonj/FLenQA/data/eval-00000-of-00001-e3c7225e84e4d34b.parquet")

# 从 'assertion/question' 列中随机选择10个不重复的值
unique_questions = dataset['assertion/question'].drop_duplicates().tolist()
selected_questions = random.sample(unique_questions, min(10, len(unique_questions)))

# 创建一个自定义的 JSON 编码器
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        return super(NumpyEncoder, self).default(obj)

# 创建QA JSON
qa_json = []
for question in selected_questions:
    row = dataset[dataset['assertion/question'] == question].iloc[0]
    qa_item = {
        "question": row['assertion/question'],
        "answer": row['facts']
    }
    qa_json.append(qa_item)

# 将QA JSON保存到文件
with open('qa_data.json', 'w', encoding='utf-8') as f:
    json.dump(qa_json, f, cls=NumpyEncoder, ensure_ascii=False, indent=2)

print("已生成QA JSON并保存到 qa_data.json 文件中。")

已生成QA JSON并保存到 qa_data.json 文件中。


In [4]:
#model_name = "/kaggle/input/llama-2/pytorch/7b-chat-hf/1"
#main(model_name, "/kaggle/working/")

In [5]:
# ... 现有代码 ...

def test_main():
    # 创建一个临时的测试数据集文件
    test_dataset = [
        {"question": "你好", "answer_matching_behavior": "你好"},
    ]
    test_dataset_path = "test_dataset.json"
    with open(test_dataset_path, "w") as f:
        json.dump(test_dataset, f)

    # 调用main函数
    model_name = "/kaggle/input/llama-2/pytorch/7b-chat-hf/1"  # 使用一个较小的模型进行测试
    ffn_activations_data, hidden_states_data = main(model_name, test_dataset_path)
    
    # 确保生成了图表
    assert os.path.exists("ffn_activations.png"), "FFN激活图未生成"
    assert os.path.exists("hidden_states.png"), "隐藏状态图未生成"

    # 清理临时文件
    os.remove(test_dataset_path)

    print("测试完成,图表已生成")
    return ffn_activations_data,hidden_states_data


In [6]:
#!rm -r /kaggle/working/CAA

In [7]:
import pandas as pd
dataset = pd.read_parquet("hf://datasets/alonj/FLenQA/data/eval-00000-of-00001-e3c7225e84e4d34b.parquet")

In [8]:
import pandas as pd
import json
import numpy as np

# 读取Parquet文件
dataset = pd.read_parquet("hf://datasets/alonj/FLenQA/data/eval-00000-of-00001-e3c7225e84e4d34b.parquet")

# 自定义JSON编码器
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        return super(NumpyEncoder, self).default(obj)

# 准备JSON数据，同时去重
json_data = []
seen_questions = set()

for _, row in dataset.iterrows():
    question = row["assertion/question"]
    if question not in seen_questions:
        seen_questions.add(question)
        json_data.append({
            "question": question,
            "answer_matching_behavior": row["facts"]
        })

# 将数据写入JSON文件
with open("test.json", "w", encoding="utf-8") as f:
    json.dump(json_data, f, ensure_ascii=False, indent=4, cls=NumpyEncoder)

print(f"已成功生成test.json文件，包含 {len(json_data)} 条去重后的数据")

已成功生成test.json文件，包含 262 条去重后的数据


In [9]:
!ls

qa_data.json  test.json


In [10]:
import json
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
import argparse
import os

class LlamaWrapper:
    def __init__(self, model_name):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name,device_map='auto',torch_dtype=torch.float16)
        
        # Register hooks for all layers
        self.ffn_activations = {}
        self.hidden_states = {}
        for i, layer in enumerate(self.model.model.layers):
            layer.mlp.act_fn.register_forward_hook(self.get_ffn_activation(i))
            layer.register_forward_hook(self.get_hidden_state(i))
    
    def get_ffn_activation(self, layer_num):
        def hook(module, input, output):
            self.ffn_activations[layer_num] = output.detach()
        return hook

    def get_hidden_state(self, layer_num):
        def hook(module, input, output):
            self.hidden_states[layer_num] = output[0].detach()
        return hook
    
    def generate(self, input_ids, max_new_tokens=50):
        generated = []
        ffn_activations_list = []
        hidden_states_list = []
        
        for _ in range(max_new_tokens):
            with torch.no_grad():
                outputs = self.model(input_ids)
            
            next_token = outputs.logits[:, -1, :].argmax(dim=-1)
            generated.append(next_token.item())
            input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=-1)
            
            ffn_activations_list.append({layer: self.ffn_activations[layer][:, -1, :] for layer in self.ffn_activations})
            hidden_states_list.append({layer: self.hidden_states[layer][:, -1, :] for layer in self.hidden_states})
            
            if next_token.item() == self.tokenizer.eos_token_id:
                break
        
        return generated, ffn_activations_list, hidden_states_list
    
    def reset_all(self):
        self.ffn_activations = {}
        self.hidden_states = {}

def process_dataset(model, tokenizer, dataset_path):
    with open(dataset_path, 'r') as f:
        dataset = json.load(f)
    
    ffn_activations_data = []
    hidden_states_data = []
    
    for item in tqdm(dataset, desc="Processing prompts"):
        input_ids = tokenizer(item['question'], return_tensors="pt").input_ids.to(model.device)
        
        model.reset_all()
        generated, ffn_activations_list, hidden_states_list = model.generate(input_ids)
        
        generated_text = tokenizer.decode(generated)
        #ddddddddddddddddddd
        #if item['answer_matching_behavior'] in generated_text:
        ffn_activations_data.extend(ffn_activations_list)
        hidden_states_data.extend(hidden_states_list)
    
    return ffn_activations_data, hidden_states_data

def plot_ffn_activations(ffn_activations_data,output_path):
    layers = list(ffn_activations_data[0].keys())
    mean_activations = {layer: torch.stack([data[layer].mean() for data in ffn_activations_data]) for layer in layers}
    
    plt.figure(figsize=(12, 6))
    for layer in layers:
        plt.plot(mean_activations[layer].cpu().numpy(), label=f'Layer {layer}')
    
    plt.xlabel('Generation Step')
    plt.ylabel('Mean FFN Activation')
    plt.title('Mean FFN Activations Across Layers During Generation')
    plt.legend()
    plt.savefig(f'{output_path}_ffn_activations.png')
    plt.close()

def plot_hidden_states(hidden_states_data,output_path):
    layers = list(hidden_states_data[0].keys())
    mean_hidden_states = {layer: torch.stack([data[layer].mean() for data in hidden_states_data]) for layer in layers}
    
    plt.figure(figsize=(12, 6))
    for layer in layers:
        plt.plot(mean_hidden_states[layer].cpu().numpy(), label=f'Layer {layer}')
    
    plt.xlabel('Generation Step')
    plt.ylabel('Mean Hidden State Value')
    plt.title('Mean Hidden State Values Across Layers During Generation')
    plt.legend()
    plt.savefig(f'{output_path}_hidden_states.png')
    plt.close()

import os
import json
from tqdm import tqdm

# ... 现有代码 ...

# ... 现有代码 ...

def main(model_name, output_dir):
    model = LlamaWrapper(model_name)
    root_dir = ""
    os.makedirs(output_dir, exist_ok=True)

    json_files = ['test.json']
    '''
    for root, dirs, files in os.walk(root_dir):
        for file in files:
            if file.endswith('.json'):
                json_files.append(os.path.join(root, file))
    '''
    for json_file in tqdm(json_files, desc="处理JSON文件"):
        successful_plots = 0
        try:
            with open(json_file, 'r') as f:
                dataset = json.load(f)

            for item in tqdm(dataset, desc="处理数据项"):
                input_ids = model.tokenizer(item['question'], return_tensors="pt").input_ids.to(model.device)

                model.reset_all()
                generated, ffn_activations_list, hidden_states_list = model.generate(input_ids)

                # 生成唯一的文件名
                base_name = os.path.splitext(os.path.basename(json_file))[0]
                ffn_plot_name = f"ffn_activations_{base_name}_{successful_plots+1}.png"
                hidden_plot_name = f"hidden_states_{base_name}_{successful_plots+1}.png"
                try:

                    plot_ffn_activations(ffn_activations_list, os.path.join(output_dir ,ffn_plot_name))
                    plot_hidden_states(hidden_states_list, os.path.join(output_dir, hidden_plot_name))

                    successful_plots += 1
                    print(f"成功处理数据项: {item['question'][:50]}...")

                except:
                    pass
                if successful_plots >= 10:
                    print("已生成10张图，停止处理")
                    break
        except Exception as e:
            print(f"处理文件 {json_file} 时出错: {str(e)}")
        
    print(f"总共成功生成了 {successful_plots} 张图")

# ... 其他代码保持不变 ...

In [ ]:
model_name = "/kaggle/input/llama-2/pytorch/7b-hf/1"
main(model_name, "/kaggle/working/")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/opt/conda/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:606: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/transformers/generation/configuratio

成功处理数据项: Is Ethan Washington in a marble-floored room?...



处理数据项:   1%|          | 2/262 [00:24<48:17, 11.14s/it]  

成功处理数据项: Is Christina Garcia in a blue walled room?...
